# 🧹 Module 3 — Class 1: Clean Real-World Customer Data

**Lesson:** [https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1)

---

## 📖 Today's Story

It is Monday morning. You work at a telecom company in Tashkent.

Your boss walks in:

> *"We are losing customers. Tell me who will leave next month. Use our customer data."*

You ask the data team for the file. They send you `customers.csv` and say:

> *"This file came from 3 different systems — billing, the website signup form, and the call-center logs. We merged them all together last week. Some fields are missing. Some values look weird. Some customers are in there twice. Sorry — that's what we have. Clean it."*

**This is real life.** Real customer data is **always** messy. Today we clean. Tomorrow we train.

---

## 💡 Why is cleaning important?

**Bad data → bad model → wrong answer → angry boss.**

Real data scientists spend **80%** of their time cleaning data. Not training models. Cleaning.

If you can clean data well, you can do ML.

---

## 🤖 Notebook basics

Two kinds of cells:
- 📝 **Text cells** — they explain things.
- 💻 **Code cells** — Python code. Run with **Shift + Enter**.

## 📦 Read the comments!

Comments start with `#`. Python skips them. They are notes for YOU. **Read every comment.**

## 📋 Today's plan

1. **Generate** a realistic dirty customer file (like the data team would send)
2. **Look** at the data
3. **Fix** the type problems
4. **Fill** the empty values
5. **Standardize** the text
6. **Remove** duplicates
7. **Handle** outliers
8. **Save** the clean version
9. **Bonus:** apply what we learned to a real public dataset (Telco Churn) — so we have data for next class

Let's go! 🚀

---
## Step 0: Get the Tools

We need 4 helpers (libraries):
- **pandas** = tables
- **numpy** = numbers
- **matplotlib + seaborn** = charts

In [ ]:
# 'import' = bring in the tool. 'as' = give it a short name.
import pandas as pd               # tables
import numpy as np                # numbers
import matplotlib.pyplot as plt   # charts
import seaborn as sns             # nicer charts

import warnings
warnings.filterwarnings('ignore')

print("Tools ready!")

---
## Step 1: Generate Today's Dirty Dataset

### 💡 Why generate it?

Real customer data is **private** — we can't share Uztelecom's actual customers in a public class. So we **build a fake-but-realistic file** that looks like a real merge from 3 systems.

**Important:** the dirt is real (missing values, typos, duplicates, broken types). Only the names are fake.

Don't worry about HOW we generate the data — just run the cell. Focus on the cleaning steps that come after.

In [ ]:
# ─────────────────────────────────────────────────
# Generate a realistic dirty customer file.
# Same seed = everyone gets the same data.
# ─────────────────────────────────────────────────
rng = np.random.default_rng(42)
n = 3000  # number of customers

# Pools of realistic values
first_names = ['Ali', 'Zara', 'Jasur', 'Madina', 'Otabek', 'Aisha', 'Karim',
               'Nargiza', 'Bobur', 'Dilnoza', 'Sardor', 'Malika', 'Timur',
               'Shaxnoza', 'Rustam', 'Feruza']
cities       = ['Tashkent', 'Samarkand', 'Bukhara', 'Andijan', 'Namangan', 'Fergana']
plans        = ['basic', 'pro', 'premium']
internet     = ['Fiber', 'DSL', 'No internet service']

# Build the clean version first
df = pd.DataFrame({
    'customer_id':    [f"CU{i:05d}" for i in range(1, n + 1)],
    'name':           rng.choice(first_names, n),
    'city':           rng.choice(cities, n),
    'age':            rng.integers(18, 75, n),
    'tenure_months':  rng.integers(1, 73, n).astype(float),
    'monthly_charges': rng.normal(80, 25, n).round(2),
    'plan':           rng.choice(plans, n),
    'internet_service': rng.choice(internet, n, p=[0.45, 0.35, 0.20]),
    'has_phone_service': rng.choice(['Yes', 'No phone service'], n, p=[0.9, 0.1]),
    'churn':          rng.choice(['Yes', 'No'], n, p=[0.27, 0.73])
})
df['total_charges'] = (df['tenure_months'] * df['monthly_charges']).round(2)

# ─── Now we add REAL dirt — the kind a 3-system merge produces ───

# (1) Missing values in tenure_months and monthly_charges
df.loc[rng.choice(n, 80, replace=False), 'tenure_months']  = np.nan
df.loc[rng.choice(n, 60, replace=False), 'monthly_charges'] = np.nan
df.loc[rng.choice(n, 40, replace=False), 'age']             = np.nan

# (2) total_charges stored as STRING (CSV export gotcha)
df['total_charges'] = df['total_charges'].astype(str)
df.loc[rng.choice(n, 25, replace=False), 'total_charges'] = ' '   # blank string

# (3) Inconsistent capitalization in city (4 systems = 4 spellings)
tashkent_idx = df.index[df['city'] == 'Tashkent'].tolist()
rng.shuffle(tashkent_idx)
df.loc[tashkent_idx[:80],   'city'] = 'tashkent'
df.loc[tashkent_idx[80:140],'city'] = 'TASHKENT'
df.loc[tashkent_idx[140:200],'city'] = ' Tashkent '

# (4) Inconsistent capitalization in plan
pro_idx = df.index[df['plan'] == 'pro'].tolist()
rng.shuffle(pro_idx)
df.loc[pro_idx[:50],   'plan'] = 'PRO'
df.loc[pro_idx[50:80], 'plan'] = 'Pro '

# (5) Inconsistent yes/no in churn
yes_idx = df.index[df['churn'] == 'Yes'].tolist()
rng.shuffle(yes_idx)
df.loc[yes_idx[:30],   'churn'] = 'yes'
df.loc[yes_idx[30:60], 'churn'] = 'YES'
df.loc[yes_idx[60:90], 'churn'] = ' Yes '

# (6) Impossible values (data entry errors)
df.loc[100, 'tenure_months']  = -3      # negative
df.loc[200, 'tenure_months']  = 999     # 999 months = 83 years
df.loc[300, 'monthly_charges'] = 99999  # billing error
df.loc[400, 'age']            = 200     # 200 years old

# (7) Duplicate customers (the merge from 3 systems)
df = pd.concat([df, df.iloc[[5, 50, 100, 500, 1000, 2000]]], ignore_index=True)

# Save to a CSV file (so we 'received' the file from the data team)
df.to_csv('customers_raw.csv', index=False)

print(f"Got file: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head(10)

👀 **Look at the table above.** It looks normal at first glance.

Each row = one customer. Columns:
- `customer_id` — unique ID
- `name`, `city`, `age` — basic info
- `tenure_months` — how many months they've been our customer
- `monthly_charges` — how much they pay each month
- `total_charges` — total they've paid all together
- `plan` — basic, pro, or premium
- `internet_service` — Fiber, DSL, or no internet
- `has_phone_service` — Yes / No phone service
- `churn` — did they leave? (Yes/No) ← **What we want to predict!**

Looks clean, right? **Trust me — there are problems hiding inside.** Let's find them.

---
## Step 2: Look Before You Touch

### 💡 Why?

Imagine you got a new car. **First thing?** You look around. Check the lights. Check the tires. Smell the oil.

Same with data. **First we LOOK. Then we change.**

Three commands tell us a lot:

In [ ]:
# .info() shows: column name, type, and how many values are NOT empty.
#
# Look at 'Dtype':
#   'object'  = text
#   'int64'   = whole number (5)
#   'float64' = number with a dot (19.95)
df.info()

🚨 **Already 2 problems!**

1. **`total_charges`** is `object` (text). But it's money — it should be a number!
2. **`tenure_months`, `monthly_charges`, `age`** have fewer non-null values than total rows. Some are empty!

In [ ]:
# .describe() = numbers about NUMBER columns.
# Always check min and max — bad data hides there!
df.describe()

🚨 **More problems!**

Look carefully:
- `tenure_months` min = **−3**. Negative months? Impossible!
- `tenure_months` max = **999**. 999 months = 83 years. Impossible!
- `monthly_charges` max = **99,999**. One customer pays 99,999? Real or typo?
- `age` max = **200**. 200 years old? Impossible!

These are why we look BEFORE we touch.

In [ ]:
# Empty values per column. True = empty. .sum() counts True values.
df.isnull().sum()

In [ ]:
# .value_counts() shows how many of each value.
# Look at the city column.
df['city'].value_counts()

🚨 **Look! The city column has 4 different ways to write 'Tashkent':**
- `Tashkent`
- `tashkent` (different signup form)
- `TASHKENT` (auto-caps from a phone keyboard)
- ` Tashkent ` (extra spaces from Excel)

The model thinks they are 4 different cities! We will fix this in Step 5.

### 🤔 Stop and think

1. How many rows? How many columns?
2. Which column is `object` but should be a number?
3. Which 3 columns have empty values?
4. What is the smallest tenure? Why is that a problem?

---
## Step 3: Fix the Money Column 💰

### 💡 Why?

`total_charges` is text. **You can not do math on text!**

In your head: `"hello" + "world"` = `"helloworld"` (text). But `5 + 3` = `8` (number).

If the model sees `"100.50"` (text), it crashes.

**We must change text to numbers.** Let's see why it's text:

In [ ]:
# Try to change to number. errors='coerce' = bad values become NaN.
# .isna() finds those bad values.
mask = pd.to_numeric(df['total_charges'], errors='coerce').isna()
print(f"Rows that can't become numbers: {mask.sum()}")
print("\nLet's look at them:")
df.loc[mask, ['customer_id', 'total_charges', 'tenure_months']].head()

💡 **AHA!** The bad rows have `total_charges = ' '` (a blank space, not a real value).

**Fix:**
1. Convert the column to numbers — empty space becomes NaN.
2. Fill NaN with the median (or 0, depending on what makes sense).

In [ ]:
# Step 1: change to number. Bad values become NaN.
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')

# Step 2: fill NaN with the median (safe default).
df['total_charges'] = df['total_charges'].fillna(df['total_charges'].median())

# Check
print("Type now:", df['total_charges'].dtype)
print("Empty values:", df['total_charges'].isna().sum())

---
## Step 4: Fill the Empty Values

### 💡 Why?

Real life: customer didn't fill in the form, signup system crashed, the merge lost values. **The model crashes with NaN.**

Three ways to fill:

| Way | Best for |
| --- | --- |
| **Mean** (average) | Numbers, no big outliers |
| **Median** (middle) | Numbers, safer with outliers |
| **Mode** (most common) | Text columns |

In [ ]:
# Empty values per column
missing = df.isnull().sum()
print("Empty values:")
print(missing[missing > 0])

In [ ]:
# Fill with MEDIAN (we saw a 99,999 outlier — median is safer than mean).
for col in ['tenure_months', 'monthly_charges', 'age']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Filled {col} with median = {median_val:.1f}")

# Check — should now be 0 empty values
print(f"\nEmpty values now: {df.isnull().sum().sum()}")

### 📝 YOUR TURN — missing-indicator column

Sometimes 'this value is empty' tells us something useful! A customer who didn't fill in their plan → maybe didn't trust the form → maybe will leave.

**Trick:** add a 'was_missing' column **BEFORE** you fill the values.

```python
df['tenure_was_missing'] = df['tenure_months'].isna().astype(int)
# .astype(int) changes True/False to 1/0
```

**But we already filled them.** So `isna()` returns all False now.

**Real lesson: ORDER MATTERS in cleaning.** Missing-indicators must come first.

In [ ]:
# YOUR CODE — try the indicator approach.
# Hint: re-load the raw file (customers_raw.csv), do the indicator BEFORE fillna.

# Example:
# raw = pd.read_csv('customers_raw.csv')
# raw['tenure_was_missing'] = raw['tenure_months'].isna().astype(int)
# print(raw['tenure_was_missing'].value_counts())

---
## Step 5: Same Word, Same Meaning

### 💡 Why?

We saw 4 spellings of Tashkent and inconsistent plan names. The model thinks they're different categories.

**One meaning = one word.** Always.

### 📚 The 2-step pattern

```python
df['col'] = df['col'].str.strip().str.lower()
```

- `.str.strip()` removes spaces at start and end
- `.str.lower()` makes all letters small

These 2 steps fix **most** dirty text in real datasets.

In [ ]:
# Before — count of each city
print("BEFORE:")
print(df['city'].value_counts())

# Strip spaces, then lowercase
df['city'] = df['city'].str.strip().str.lower()
df['plan'] = df['plan'].str.strip().str.lower()
df['churn'] = df['churn'].str.strip().str.lower()

print("\nAFTER:")
print(df['city'].value_counts())
print("\nPlan:")
print(df['plan'].value_counts())
print("\nChurn:")
print(df['churn'].value_counts())

In [ ]:
# Now fix the verbose 'No internet service' / 'No phone service' values.
# These really mean 'No'. We collapse 3 categories to 2.
df['internet_service'] = df['internet_service'].replace('No internet service', 'no_internet')
df['has_phone_service'] = df['has_phone_service'].replace('No phone service', 'No')

print("internet_service:", df['internet_service'].unique())
print("has_phone_service:", df['has_phone_service'].unique())

✅ Now the model sees one form per category.

Note: we kept `Fiber` / `DSL` / `no_internet` as 3 distinct categories for `internet_service` — they're really different services. But for `has_phone_service`, `'No phone service'` and `'No'` mean the same thing → collapsed to `'No'`.

---
## Step 6: Remove Duplicate Customers

### 💡 Why?

When the data team merged 3 systems, the same customer can appear **twice** (or more)!

Counting Ali twice → wrong metrics. The model trains on Ali twice → memorizes Ali too well.

**Each customer should appear ONCE.**

In [ ]:
# How many duplicate customers? (same customer_id = same customer)
dups = df.duplicated(subset=['customer_id']).sum()
print(f"Duplicate customers: {dups}")
print(f"Total rows now: {len(df)}")

# Drop duplicates. Keep the first occurrence.
df = df.drop_duplicates(subset=['customer_id']).reset_index(drop=True)

print(f"\nAfter dedupe: {len(df)} rows.")

💡 **Why dedupe by `customer_id` and not by entire row?**

If Ali was in 2 systems and one had an extra space in his name, the rows aren't identical — but it's still the same Ali. Use the **business key** (customer ID), not the whole row.

---
## Step 7: Find the Weird Numbers (Outliers)

### 💡 Why?

Remember from Step 2:
- `tenure_months = -3` → impossible (negative!)
- `tenure_months = 999` → impossible (83 years!)
- `monthly_charges = 99,999` → typo? VIP? fraud?
- `age = 200` → impossible

Three kinds of outliers:

1. **Mistakes** (typing errors, broken sensors) → fix or remove
2. **Real but very high** (a VIP customer) → keep, maybe cap
3. **The thing you want to find** (fraud!) → never remove!

🚨 **Always think first:** which kind do I have?

In [ ]:
# Step 1: Fix IMPOSSIBLE values.
# tenure must be 0 to ~120, age must be 0 to 120, charges must be positive.

impossible_tenure = (df['tenure_months'] < 0) | (df['tenure_months'] > 120)
impossible_age    = (df['age'] < 0) | (df['age'] > 120)

print(f"Impossible tenure: {impossible_tenure.sum()} → {df.loc[impossible_tenure, 'tenure_months'].tolist()}")
print(f"Impossible age   : {impossible_age.sum()} → {df.loc[impossible_age, 'age'].tolist()}")

# Replace impossible with NaN, then fill with median
df.loc[impossible_tenure, 'tenure_months'] = np.nan
df.loc[impossible_age, 'age']               = np.nan

df['tenure_months'] = df['tenure_months'].fillna(df['tenure_months'].median())
df['age']           = df['age'].fillna(df['age'].median())

In [ ]:
# Step 2: Find statistical outliers using the IQR rule.
#
# Q1   = 25% point. Q3 = 75% point. IQR = Q3 - Q1.
# outlier if value < Q1 - 1.5×IQR or > Q3 + 1.5×IQR

def detect_outliers_iqr(series, factor=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return (series < lower) | (series > upper), lower, upper

for col in ['monthly_charges', 'total_charges', 'tenure_months']:
    outliers, low, high = detect_outliers_iqr(df[col])
    print(f"{col}: {outliers.sum()} outliers (limits: [{low:.2f}, {high:.2f}])")

In [ ]:
# Cap monthly_charges to the IQR upper limit (winsorize).
# This keeps the customer but prevents the 99,999 from breaking things.
outliers, low, high = detect_outliers_iqr(df['monthly_charges'])
df['monthly_charges'] = df['monthly_charges'].clip(lower=low, upper=high)

print(f"monthly_charges max now: {df['monthly_charges'].max():.2f}")

In [ ]:
# Box plots — visualize the distributions.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(['monthly_charges', 'total_charges', 'tenure_months']):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

💡 **Lesson:** *Big number ≠ wrong number.* A 72-month tenure is just a loyal customer, not an error. Always think first.

---

## Step 8: Save the Clean Data

We just spent the whole class cleaning. **Don't lose your work!**

In [ ]:
# Final check
print("Shape:", df.shape)
print("Empty values:", df.isnull().sum().sum())
print("\nColumn types:")
print(df.dtypes.value_counts())

In [ ]:
# Save to a NEW file. Never overwrite the original!
df.to_csv('customers_cleaned.csv', index=False)
print("✅ Saved: customers_cleaned.csv")

---
## 🎁 Bonus: Apply What We Learned to a Real Dataset

### 💡 Why?

We just cleaned synthetic data. **Will the same techniques work on real public data?** YES.

We'll grab the **Telco Customer Churn** dataset (a famous public telecom dataset) and apply our cleaning skills. The output of this section is what we use in **Class 2** to train a model.

Telco is mostly clean (because it's been used in classes for years), but it still has 2 real-world problems we know how to fix.

In [ ]:
# Load the real Telco file
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(url)

print(f"Loaded {telco.shape[0]} customers, {telco.shape[1]} columns.")
telco.info()

### Two real problems in Telco

1. **`TotalCharges` is `object` (text)** — same problem we just solved! Some rows have blank space `' '` (these are brand-new customers with `tenure=0` who haven't been billed yet).

2. **`'No internet service'` and `'No phone service'`** — verbose values that really mean `'No'`. The model sees them as separate categories.

**Same techniques. Real public data.**

In [ ]:
# FIX 1: Convert TotalCharges to a number. Fill blanks with 0 (new customers paid 0).
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce').fillna(0)

# FIX 2: Replace 'No internet service' / 'No phone service' with 'No'
internet_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in internet_cols:
    telco[col] = telco[col].replace('No internet service', 'No')
telco['MultipleLines'] = telco['MultipleLines'].replace('No phone service', 'No')

# Save — this is what Class 2 will use
telco.to_csv('telco_churn_cleaned.csv', index=False)
print("✅ Saved: telco_churn_cleaned.csv")
print(f"Shape: {telco.shape}")
print(f"TotalCharges type: {telco['TotalCharges'].dtype}")
print(f"OnlineSecurity values: {telco['OnlineSecurity'].unique()}")

✅ **Now we have two cleaned datasets:**

1. `customers_cleaned.csv` — our synthetic Uztelecom data (where we did all the practice)
2. `telco_churn_cleaned.csv` — the public Telco dataset, ready for Class 2's modeling work

---
## 🏁 We Did It!

### What we cleaned today

| Step | What we found | What we did |
| --- | --- | --- |
| 2. Look | `total_charges` as text, missing values, impossible numbers | Found problems |
| 3. Fix money | Blank-string rows in `total_charges` | Made it a number |
| 4. Fill empty | Missing values in 3 columns | Filled with median |
| 5. Same words | 4 spellings of Tashkent, mixed Yes/yes/YES | One spelling per category |
| 6. Duplicates | Customers in twice from the merge | Dropped duplicates |
| 7. Outliers | tenure=-3, tenure=999, monthly_charges=99,999, age=200 | Replaced + capped |
| 8. Save | Clean file ready | Saved as `customers_cleaned.csv` |
| 9. Bonus | Applied to real Telco data | Saved `telco_churn_cleaned.csv` for Class 2 |

### 🎯 Where are we going?

**Today:** Clean data ✅

**Class 2:** Make the data ready for the model (encoding, scaling).

**Class 3:** Build new useful columns (feature engineering).

**Module 4:** Train the model. Predict who will leave!

Each class makes the next class possible. 🪜

### 🎓 Words to Remember

- **DataFrame** — a table
- **NaN** — empty value
- **dtype** — type of a column
- **Median** — middle value (safer than mean for dirty data)
- **Outlier** — a value very different from the others
- **Winsorize / clip** — cap a value to a max instead of removing it
- **Domain knowledge** — knowing the business, not just the math

### 📤 Submit

Save this notebook as `Module3_Class1_<YourName>.ipynb`.

Push to your group repo at `module-3/class_1/submissions/<YourName>/`.

🧹 *Great job! See you in Class 2!*